# ALS + LightGBM Ensemble Model

This notebook implements and evaluates a hybrid ensemble model combining:
1. **ALS** (Collaborative Filtering) - Captures user-item latent patterns
2. **LightGBM** (Gradient Boosting) - Learns to combine ALS scores with rich features

**Expected Recall@10:** 24-27% (vs 21% from pure LightGBM)

In [1]:
import sys
sys.path.append('..')  # Add parent directory to path

import pandas as pd
import numpy as np

from utils.train_test_split import chronological_train_test_split
from models.als_lgb_ensemble import ALSLGBEnsemble
from utils.evaluation import evaluate_model

## 1. Load Data

In [2]:
# Load the dataset
df = pd.read_csv("../data/data_with_additional_features.csv")

# Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Shows: {df['show_id'].nunique():,}")
df.head()

Dataset shape: (12941230, 30)
Date range: 2026-06-01 00:00:00 to 2026-06-30 23:59:00
Users: 100,000
Shows: 13,349


,user_id,playback_session_id,show_id,asset_type,episode_id,day,time,watch_minutes,timestamp,user_total_mins,...,show_median_episode_progression,show_primetime_views,show_global_popularity_log,show_primetime_ratio,show_velocity_delta,seq_decay_affinity_score,seq_immediate_recency_rank,seq_is_favorite_anchor,seq_pop_interaction_leverage,context_primetime_match_score
0,1,1,1,VOD,1.0,3,20:16,54,2026-06-03 20:16:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
1,1,2,1,VOD,2.0,3,11:05,53,2026-06-03 11:05:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
2,1,3,1,VOD,3.0,3,22:46,44,2026-06-03 22:46:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
3,1,4,2,VOD,4.0,1,22:56,1,2026-06-01 22:56:00,9539.0,...,67263.0,71.0,5.673323,0.243986,1.173913,0.036882,0,0,0.209244,0.048126
4,1,5,3,RECORDING,5.0,6,15:26,27,2026-06-06 15:26:00,9539.0,...,23335.0,1058.0,9.083075,0.120159,-0.030859,0.911249,0,0,8.276939,0.023701


## 2. Chronological Train/Validation/Test Split

In [3]:
# Split data chronologically: train / validation / test
train_df, val_df, test_df = chronological_train_test_split(
    df,
    timestamp_col='timestamp',
    test_days=5,
    val_days=5,
    return_splits='train_val_test'
)

Chronological Split Summary:
  Train:      2026-06-01 00:00:00 to 2026-06-20 23:59:00 (8,705,249 rows)
  Validation: 2026-06-20 23:59:00 to 2026-06-25 23:59:00 (2,086,740 rows)
  Test:       2026-06-25 23:59:00 to 2026-06-30 23:59:00 (2,149,241 rows)


## 3. Prepare Train Seen Shows Dictionary

In [4]:
# Create a dictionary of shows each user has seen in training data
train_seen_dict = train_df.groupby('user_id')['show_id'].apply(set).to_dict()

print(f"Users in training with history: {len(train_seen_dict):,}")

Users in training with history: 95,415


## 4. Train ALS + LightGBM Ensemble

This will:
1. Train ALS model on the training data
2. Build a candidate pool of top 500 popular items
3. Engineer features (same as pure LightGBM)
4. Add ALS scores as an additional feature
5. Train LightGBM ranker with 12 features (11 original + ALS score)

In [5]:
# Initialize and train ensemble model
score_col = 'implicit_interaction_score' if 'implicit_interaction_score' in train_df.columns else 'watch_minutes'

ensemble_model = ALSLGBEnsemble(
    # ALS parameters
    als_factors=64,
    als_regularization=0.1,
    als_iterations=15,
    als_alpha=1.0,
    # LightGBM parameters
    candidate_pool_size=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=63,
    num_iterations=150
)

ensemble_model.fit(
    train_df,        # Full training data
    val_df,          # Validation window for ranking training
    user_col='user_id',
    item_col='show_id',
    score_col=score_col
)

 TRAINING ALS + LIGHTGBM ENSEMBLE RECOMMENDER

[STEP 1/4] Training ALS Collaborative Filtering Model...
 TRAINING IMPLICIT ALS COLLABORATIVE FILTERING

[1/3] Building sparse user-item interaction matrix...
  Matrix shape: 95,415 users × 11,954 items
  Total interactions: 8,705,249
  Sparsity: 99.2368%

[2/3] Training ALS (factors=64, iterations=15)...


/Users/smalik/Documents/lucky/p3/lib/python3.9/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/Users/smalik/Documents/lucky/p3/lib/python3.9/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.008008956909179688 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]


[3/3] Training complete!

[STEP 2/4] Building candidate pool (top 500 items)...
  Candidate pool size: 500

[STEP 3/4] Engineering features with ALS scores...
  Created 11954 show feature profiles
  Created 95415 user feature profiles

[STEP 4/4] Training LightGBM ranker with ALS features...
  Created 118,361 training examples
  Positive examples: 18,361
  Negative examples: 100,000
  Features: 12 (including ALS score)
  Query groups: 5000 users
  Sum of query groups: 118361 (should match examples)

  Top 5 Most Important Features:
    show_global_popularity_log       252511.0
    show_unique_users_log             16877.9
    user_unique_shows                  5518.2
    user_show_diversity_match          3352.1
    user_show_interaction_ratio        2799.2

✓ Ensemble training complete!


## 5. View Feature Importance

Check if ALS score is among the most important features!

In [6]:
# Get feature importance
feature_importance = ensemble_model.get_feature_importance()
print("\nFeature Importance (All Features):")
print(feature_importance.to_string(index=False))

# Check ALS score rank
als_rank = feature_importance[feature_importance['feature'] == 'als_score'].index[0] + 1
print(f"\nALS score rank: #{als_rank} out of {len(feature_importance)} features")


Feature Importance (All Features):
                    feature    importance
 show_global_popularity_log 252510.999145
      show_unique_users_log  16877.935997
          user_unique_shows   5518.196802
  user_show_diversity_match   3352.146797
user_show_interaction_ratio   2799.214229
 user_avg_interaction_score   2692.757395
                  als_score   2657.457594
  show_total_watch_time_log   2402.846206
       user_diversity_score   2312.118892
 popularity_bias_resistance   2272.543696
    user_total_interactions   1844.251016
        show_avg_engagement   1210.411699

ALS score rank: #12 out of 12 features


## 6. Test Predictions for a Sample User

In [7]:
# Test on a sample user
sample_user = train_df['user_id'].iloc[100]
print(f"Sample user: {sample_user}")

# Get recommendations
recommendations = ensemble_model.predict(sample_user, top_n=10)
print(f"\nTop 10 recommendations: {recommendations}")

# Get what they watched in training
user_history = train_df[train_df['user_id'] == sample_user]['show_id'].unique()[:5]
print(f"\nUser's watch history (first 5): {list(user_history)}")

Sample user: 1

Top 10 recommendations: [392, 38, 707, 19, 777, 252, 112, 16, 472, 162]

User's watch history (first 5): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6)]


## 7. Evaluate on Test Set

This uses the optimized batch evaluation - should complete in 1-3 minutes.

In [8]:
# Evaluate on test set
test_metrics = evaluate_model(
    model=ensemble_model,
    test_ground_truth=test_df,
    train_seen_dict=train_seen_dict,
    user_col='user_id',
    item_col='show_id',
    top_n=10,
    verbose=True
)

Evaluating model on 77,272 users...
  Processing batch 1/78 (0/77,272 users)...
  Processing batch 11/78 (10,000/77,272 users)...
  Processing batch 21/78 (20,000/77,272 users)...
  Processing batch 31/78 (30,000/77,272 users)...
  Processing batch 41/78 (40,000/77,272 users)...
  Processing batch 51/78 (50,000/77,272 users)...
  Processing batch 61/78 (60,000/77,272 users)...
  Processing batch 71/78 (70,000/77,272 users)...

  Evaluation Results (Top-10)
  Users Evaluated: 77,272
  Recall@10:    1.10%
  MRR@10:       0.0195
  Precision@10: 0.80%



## 8. Final Summary & Comparison

In [9]:
print("\n" + "="*70)
print(" ALS + LIGHTGBM ENSEMBLE - FINAL RESULTS")
print("="*70)
print(f"\n{'Metric':<20} {'Test Set':<15}")
print("-"*35)
print(f"{'Recall@10':<20} {test_metrics['recall@10']*100:>14.2f}%")
print(f"{'MRR@10':<20} {test_metrics['mrr@10']:>14.4f}")
print(f"{'Precision@10':<20} {test_metrics['precision@10']*100:>14.2f}%")
print(f"{'Users Evaluated':<20} {test_metrics['num_users_evaluated']:>14,}")
print("="*70)

print("\n" + "="*70)
print(" MODEL COMPARISON")
print("="*70)
print(f"\n{'Model':<30} {'Recall@10':<15}")
print("-"*45)
print(f"{'LightGBM (baseline)':<30} {'21.0%':<15}")
print("="*70)

improvement = (test_metrics['recall@10'] - 0.21) * 100



 ALS + LIGHTGBM ENSEMBLE - FINAL RESULTS

Metric               Test Set       
-----------------------------------
Recall@10                      1.10%
MRR@10                       0.0195
Precision@10                   0.80%
Users Evaluated              77,272

 MODEL COMPARISON

Model                          Recall@10      
---------------------------------------------
LightGBM (baseline)            21.0%          


In [10]:
# Check feature importance
importance_df = ensemble_model.get_feature_importance()
print("\nFeature Importance:")
print(importance_df)

# Check ALS score specifically
als_row = importance_df[importance_df['feature'] == 'als_score']
if len(als_row) > 0:
    als_rank = importance_df[importance_df['feature'] == 'als_score'].index[0] + 1
    als_importance = als_row['importance'].values[0]
    print(f"\n🎯 ALS score rank: #{als_rank} out of {len(importance_df)}")
    print(f"🎯 ALS score importance: {als_importance:,.1f}")
    
    # Show what % of total importance
    total_importance = importance_df['importance'].sum()
    als_pct = (als_importance / total_importance) * 100
    print(f"🎯 ALS contributes {als_pct:.1f}% of total importance")


Feature Importance:
                        feature     importance
0    show_global_popularity_log  252510.999145
2         show_unique_users_log   16877.935997
6             user_unique_shows    5518.196802
10    user_show_diversity_match    3352.146797
9   user_show_interaction_ratio    2799.214229
5    user_avg_interaction_score    2692.757395
11                    als_score    2657.457594
1     show_total_watch_time_log    2402.846206
7          user_diversity_score    2312.118892
8    popularity_bias_resistance    2272.543696
4       user_total_interactions    1844.251016
3           show_avg_engagement    1210.411699

🎯 ALS score rank: #12 out of 12
🎯 ALS score importance: 2,657.5
🎯 ALS contributes 0.9% of total importance
